# Product Demand Forecasting – EDA

Exploratory data analysis using pandas.

## 1. Load Data

Two options below – use whichever fits your setup.

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [5]:
# ── Option A: load from Kaggle directly ──────────────────────────────────────
# Run this block first if you have a kaggle.json API token

# from google.colab import files
# files.upload()   # select your kaggle.json

# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d felixzhao/productdemandforecasting --unzip -p data/

# ── Option B: upload from laptop ─────────────────────────────────────────────
# from google.colab import files
# import io

# print("Upload your CSV file:")
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# DATA_PATH = fname
# print("Loaded:", fname)

df_raw = pd.read_csv('Historical Product Demand.csv')

In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# If you used Option A (Kaggle), set this manually:
# DATA_PATH = "data/Historical Product Demand.csv"

# df_raw = pd.read_csv(io.BytesIO(uploaded[fname]))
# print("Raw shape:", df_raw.shape)
# df_raw.head()

## 2. Clean Data

In [7]:
df = df_raw.copy()
df.columns = [c.strip() for c in df.columns]

# parse date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# fix demand column — some values are like (1000) meaning negative
df['Order_Demand'] = (
    df['Order_Demand'].astype(str)
    .str.replace(r'[()\s,]', '', regex=True)
    .replace('nan', np.nan)
)
df['Order_Demand'] = pd.to_numeric(df['Order_Demand'], errors='coerce')
df = df.dropna(subset=['Order_Demand'])
df['Order_Demand'] = df['Order_Demand'].astype(int)

for col in ['Product_Code', 'Warehouse', 'Product_Category']:
    df[col] = df[col].astype(str).str.strip()

df = df.sort_values('Date').reset_index(drop=True)
print("Clean shape:", df.shape)
df.info()

Clean shape: (1048575, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype         
---  ------            --------------    -----         
 0   Product_Code      1048575 non-null  object        
 1   Warehouse         1048575 non-null  object        
 2   Product_Category  1048575 non-null  object        
 3   Date              1037336 non-null  datetime64[ns]
 4   Order_Demand      1048575 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 40.0+ MB


## 3. Basic Statistics

In [8]:
print("Date range  :", df['Date'].min(), "to", df['Date'].max())
print("Warehouses  :", df['Warehouse'].nunique(), "–", sorted(df['Warehouse'].unique()))
print("Categories  :", df['Product_Category'].nunique())
print("Products    :", df['Product_Code'].nunique())
print("Total demand:", f"{df['Order_Demand'].sum():,}")
print()
df['Order_Demand'].describe().round(2)

Date range  : 2011-01-08 00:00:00 to 2017-01-09 00:00:00
Warehouses  : 4 – ['Whse_A', 'Whse_C', 'Whse_J', 'Whse_S']
Categories  : 33
Products    : 2160
Total demand: 5,145,333,321



count    1048575.00
mean        4906.98
std        28926.78
min            0.00
25%           20.00
50%          300.00
75%         2000.00
max      4000000.00
Name: Order_Demand, dtype: float64

## 4. Missing Values

In [9]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'count': missing, 'pct': missing_pct}).query('count > 0')

,count,pct
Date,11239,1.07


## 5. Demand by Warehouse

In [10]:
by_wh = (df.groupby('Warehouse')['Order_Demand']
           .agg(['sum','mean','count'])
           .rename(columns={'sum':'total','mean':'avg','count':'orders'})
           .sort_values('total', ascending=False))
by_wh['total'] = by_wh['total'].map('{:,.0f}'.format)
by_wh['avg']   = by_wh['avg'].map('{:,.1f}'.format)
by_wh

,total,avg,orders
Warehouse,,,
Whse_J,"3,363,200,396","4,399.5",764447
Whse_S,"1,038,024,700","11,769.0",88200
Whse_C,"585,071,404","13,813.8",42354
Whse_A,"159,036,821","1,035.6",153574


## 6. Demand by Product Category

In [11]:
by_cat = (df.groupby('Product_Category')['Order_Demand']
            .agg(['sum','mean','count'])
            .rename(columns={'sum':'total','mean':'avg','count':'orders'})
            .sort_values('total', ascending=False))
by_cat

,total,avg,orders
Product_Category,,,
Category_019,4251207605,8836.450720,481099
Category_006,405579330,11400.043005,35577
Category_005,199681320,1963.994846,101671
Category_007,128691531,1561.752518,82402
Category_028,49150112,1570.190787,31302
Category_033,42610000,23044.889129,1849
Category_030,40966555,3152.000846,12997
Category_021,4480660,86.148315,52011
Category_032,4473048,481.179862,9296


## 7. Top 15 Products by Total Demand

In [12]:
top_products = (df.groupby('Product_Code')['Order_Demand']
                  .sum()
                  .sort_values(ascending=False)
                  .head(15)
                  .reset_index())
top_products.columns = ['Product_Code', 'Total_Demand']
top_products

,Product_Code,Total_Demand
0,Product_1359,472474000
1,Product_1248,289117000
2,Product_0083,210651000
3,Product_1341,169777000
4,Product_1295,123303000
5,Product_1241,117741000
6,Product_1245,103537000
7,Product_1286,101566400
8,Product_1432,97385000
9,Product_1274,92831000


## 8. Time Features

In [13]:
df['year']    = df['Date'].dt.year
df['month']   = df['Date'].dt.month
df['quarter'] = df['Date'].dt.quarter

monthly = (df.dropna(subset=['Date'])
             .groupby(['year','month'])['Order_Demand']
             .sum()
             .reset_index())
monthly['period'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly.tail(12)

,year,month,Order_Demand,period
56,2016.0,2.0,74065041,2016-02-01
57,2016.0,3.0,93303910,2016-03-01
58,2016.0,4.0,79503364,2016-04-01
59,2016.0,5.0,80299593,2016-05-01
60,2016.0,6.0,84553011,2016-06-01
61,2016.0,7.0,88439936,2016-07-01
62,2016.0,8.0,80471772,2016-08-01
63,2016.0,9.0,77698896,2016-09-01
64,2016.0,10.0,84000757,2016-10-01
65,2016.0,11.0,90128568,2016-11-01


## 9. Correlation & Pivot

In [14]:
pivot = df.pivot_table(
    index='Product_Category',
    columns='Warehouse',
    values='Order_Demand',
    aggfunc='sum',
    fill_value=0
)
pivot

Warehouse,Whse_A,Whse_C,Whse_J,Whse_S
Product_Category,,,,
Category_001,1749,60106,1623054,72564
Category_002,0,0,0,628
Category_003,131607,0,39267,222076
Category_004,0,0,0,99046
Category_005,1131100,12528700,128924470,57097050
Category_006,11621427,33131500,318766710,42059693
Category_007,2839680,7179524,112157242,6515085
Category_008,1903,317,15348,0
Category_009,920903,251031,446469,2163738


## 10. Save Cleaned Data

In [15]:
import os
os.makedirs('data', exist_ok=True)
df.to_csv('data/demand_clean.csv', index=False)
print("Saved → data/demand_clean.csv")

Saved → data/demand_clean.csv
